In [1]:
import pandas as pd
import numpy as np
import sqlite3

# 1. Generate Transactional Data
np.random.seed(42)
# Simulating 500 orders across 200 unique customers to guarantee repeat purchases
customer_ids = np.random.randint(1000, 1200, size=500)
order_ids = np.arange(1, 501)
sales = np.round(np.random.uniform(20, 500, size=500), 2)

df_orders = pd.DataFrame({'OrderID': order_ids, 'CustomerID': customer_ids, 'SalesValue': sales})

# 2. Load into SQLite for Analysis
conn = sqlite3.connect(':memory:')
df_orders.to_sql('orders', conn, index=False)

# 3. SQL Query 1: Segment Table & Repeat Rate
query_segments = """
WITH CustomerOrders AS (
    SELECT CustomerID, COUNT(OrderID) as OrderCount
    FROM orders
    GROUP BY CustomerID
),
CustomerClassification AS (
    SELECT
        CustomerID,
        CASE WHEN OrderCount = 1 THEN 'First-Time' ELSE 'Repeat' END as CustomerType
    FROM CustomerOrders
)
SELECT
    CustomerType,
    COUNT(CustomerID) as Total_Customers,
    ROUND(COUNT(CustomerID) * 100.0 / (SELECT COUNT(*) FROM CustomerClassification), 2) as Percentage_of_Base
FROM CustomerClassification
GROUP BY CustomerType;
"""

# 4. SQL Query 2: First-Time vs Repeat AOV
query_aov = """
WITH CustomerOrders AS (
    SELECT CustomerID, COUNT(OrderID) as OrderCount
    FROM orders
    GROUP BY CustomerID
)
SELECT
    CASE WHEN c.OrderCount = 1 THEN 'First-Time' ELSE 'Repeat' END as CustomerType,
    COUNT(o.OrderID) as Total_Transactions,
    ROUND(AVG(o.SalesValue), 2) as Average_Order_Value
FROM orders o
JOIN CustomerOrders c ON o.CustomerID = c.CustomerID
GROUP BY CustomerType;
"""

print("--- CUSTOMER SEGMENTS & REPEAT RATE ---")
print(pd.read_sql(query_segments, conn))
print("\n--- AVERAGE ORDER VALUE (AOV) COMPARISON ---")
print(pd.read_sql(query_aov, conn))

--- CUSTOMER SEGMENTS & REPEAT RATE ---
  CustomerType  Total_Customers  Percentage_of_Base
0   First-Time               41               22.78
1       Repeat              139               77.22

--- AVERAGE ORDER VALUE (AOV) COMPARISON ---
  CustomerType  Total_Transactions  Average_Order_Value
0   First-Time                  41               281.23
1       Repeat                 459               260.99
